## 0. 準備環境（直接執行，不必逐行看懂）
先依序執行下面兩格，安裝所需套件。環境已可用時會自動跳過。

**Colab 請分開執行。** 第一格安裝 Conda 後可能自動重啟；等重新連線，再執行第二格安裝 PyGMT 等套件。勿在安裝期間重複按執行。

此教材使用 PyGMT 0.17 / GMT 6.5。新的 Colab 執行環境仍需安裝。


In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, ipywidgets
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    print("步驟 1/2：安裝 Conda（約 1 分鐘）。完成後 Colab 會自動重啟執行環境，等重新連線再執行下一格。", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")


In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, ipywidgets
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    print("步驟 2/2：安裝 PyGMT 與相依套件（約 2–4 分鐘），下方會逐行顯示進度。", flush=True)
    command = ["mamba", "install", "-y", "-c", "conda-forge", "pygmt=0.17", "gmt=6.5", "ghostscript=10.04", "ipywidgets"]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
    if process.returncode != 0:
        raise RuntimeError(f"安裝失敗（exit {process.returncode}），請重新執行本格或重啟執行環境。")
    print("安裝完成，可以往下執行。")
else:
    print("跳過 Colab 安裝。")


# 02｜地形與 3D

請先另存副本，完成環境設置後，由上往下執行。

[課前介紹](https://github.com/jimmy60504/pygmt-map-lab/blob/main/intro.md) · [課程首頁](https://github.com/jimmy60504/pygmt-map-lab)

[1｜基本地圖與地震](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/01_maps_earthquakes.ipynb) · [2｜地形與 3D](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/02_terrain_3d.ipynb) · [3｜AI 探索與作業](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/03_ai_exploration.ipynb)


### 常用快捷鍵

| 操作 | Windows／Linux | Mac |
| --- | --- | --- |
| 執行目前儲存格並移到下一格 | Shift + Enter | Shift + Enter |
| 取消／切換註解 | Ctrl + / | ⌘ + / |

把游標放在程式行，或選取多行，再按切換註解的快捷鍵，即可移除或加上行首的 `#`。只選程式行，不要連中文說明一起取消註解。

修改後按 **Shift + Enter** 看結果，等執行完成再繼續下一步。


## 3. 彩色地形：一層一層看見起伏

### 第一張：平面彩色地形

先只用顏色表達高程。`02m` 是 2 角分，不是 2 公尺；首次執行會下載資料。

**查 API 玩看看**：地形能不能更細？有哪些解析度可選？改完比較網格大小、下載時間與細節。之後兩張沿用這份 `grid`，更換解析度後請依序重跑。

| 指令／官方 API | 用途 | 帶著什麼問題去查 |
| --- | --- | --- |
| [load_earth_relief()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.datasets.load_earth_relief.html) | 載入地形網格 | 如何選擇解析度與下載範圍？ |
| [fig.grdimage()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdimage.html) | 高程上色 | 如何選擇色票？ |
| [fig.colorbar()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.colorbar.html) | 顯示高程色條 | 如何標示高度單位？ |


In [ ]:
import pygmt

# 1. 取得地形網格
grid = pygmt.datasets.load_earth_relief(
    resolution="02m",  # 新增：網格間距 2 角分，不是 2 公尺
    region=[119, 123, 21, 26],  # 下載範圍：西、東、南、北
    registration="gridline",  # 網格值位於格線交點，這裡先沿用
)

print("地形網格：", grid.shape, "；高程單位：m")


# 2. 先只用高程上色，不加陰影或等高線
fig = pygmt.Figure()

fig.grdimage(
    grid=grid,  # 新增：輸入整片地形網格
    region=[119, 123, 21, 26],
    projection="M15c",
    cmap="geo",  # 改動：使用海底與陸地的地形色票
    frame=["af", "+tTaiwan: land and seafloor"],
)


fig.coast(shorelines="0.5p,gray25", resolution="h")
fig.basemap(map_scale="jBR+c24+w50k+o0.5c/0.5c+f+lkm")  # 50 km 比例尺，參考緯度 24°N
fig.colorbar(frame=["xaf", "y+lElevation (m)"])
fig.show()


### 第二張：加上陰影

同一份地形與色票，只新增模擬照光。比較第一張：山稜與谷地是否更容易辨認？陰影不是高度本身，不能只靠明暗判斷高低。

**查 API 玩看看**：陰影能不能關掉或換個角度？如果逐一改數字太麻煩，可以請 AI 把照光角度做成拉桿。調整角度時可沿用已下載的地形。

| 指令／官方 API | 用途 | 帶著什麼問題去查 |
| --- | --- | --- |
| [fig.grdimage()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdimage.html) | 在彩色地形加入陰影 | 如何控制照光方向與陰影效果？ |


In [ ]:
import pygmt

# 沿用第一張下載的 grid，不必重新下載

fig = pygmt.Figure()

fig.grdimage(
    grid=grid,  # 新增：輸入整片地形網格
    region=[119, 123, 21, 26],
    projection="M15c",
    cmap="geo",  # 改動：使用海底與陸地的地形色票
    shading="+a-45+nt1",  # 新增：模擬照光，凸顯起伏
    frame=["af", "+tTaiwan: land and seafloor"],
)


fig.coast(shorelines="0.5p,gray25", resolution="h")
fig.basemap(map_scale="jBR+c24+w50k+o0.5c/0.5c+f+lkm")  # 50 km 比例尺，參考緯度 24°N
fig.colorbar(frame=["xaf", "y+lElevation (m)"])
fig.show()


### 第三張：再疊上等高線

保留第二張的陰影，加上等高線與高度數字，讓讀者不只靠顏色讀高度。單位是公尺，海底高程是負值。

**查 API 玩看看**：線能不能更密或更疏？哪些線要標數字？線太多時是否反而難讀？手動試過後，也可以請 AI 把想調整的設定做成拉桿。

| 指令／官方 API | 用途 | 帶著什麼問題去查 |
| --- | --- | --- |
| [fig.grdcontour()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdcontour.html) | 疊加等高線與高度標示 | 如何控制線距、標註間距與外觀？ |


In [ ]:
import pygmt

# 沿用第一張下載的 grid，不必重新下載

fig = pygmt.Figure()

fig.grdimage(
    grid=grid,  # 新增：輸入整片地形網格
    region=[119, 123, 21, 26],
    projection="M15c",
    cmap="geo",  # 改動：使用海底與陸地的地形色票
    shading="+a-45+nt1",  # 新增：模擬照光，凸顯起伏
    frame=["af", "+tTaiwan: land and seafloor"],
)


# 新增：在陰影地形上疊加等高線
fig.grdcontour(
    grid=grid,
    levels=1000,  # 新增：等高線間距（m）
    annotation=2000,  # 新增：標示高度數字的間距（m）
    pen="0.3p,gray30",  # 新增：線條外觀
)

fig.coast(shorelines="0.5p,gray25", resolution="h")
fig.basemap(map_scale="jBR+c24+w50k+o0.5c/0.5c+f+lkm")  # 50 km 比例尺，參考緯度 24°N
fig.colorbar(frame=["xaf", "y+lElevation (m)"])
fig.show()


## 4. 3D 地形：換個角度看台灣
用 `grdview()` 將同一份地形畫成斜視圖。`perspective=[方位角, 仰角]` 控制觀看方向，`zsize` 控制垂直尺寸。

圖中垂直方向為了辨認起伏而誇大，不能當成真實坡度。這是固定視角圖片，不是滑鼠可拖曳的模型。

**試看看**：把方位角 135 改為 225，仰角維持 35，觀察哪些山被遮住。

想連續比較不同視角，也可以請 AI 把參數做成拉桿。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.grdview()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdview.html) | 繪製斜視地形表面 | `grid`、`region`、`perspective`、`zsize`、`surftype` |


In [ ]:
import pygmt


# 1. 設定觀看方向
azimuth = 135  # 方位角：繞地形從哪個方向看（度）
elevation = 35  # 仰角：觀看角度的高低（度）


# 2. 沿用上一格的 grid，改畫 3D 地形
fig = pygmt.Figure()

fig.grdview(
    grid=grid,  # 沿用：同一份地形資料
    region=[119, 123, 21, 26, -8000, 4000],  # 新增：最後兩個值是高程下、上限（m）
    projection="M15c",
    perspective=[azimuth, elevation],  # 新增：方位角、仰角
    zsize="3c",  # 新增：垂直軸畫成多高；不是實際山高
    surftype="s",  # 新增：繪製表面
    cmap="geo",  # 沿用：高程色票
    frame=[
        "xaf",  # x 軸刻度
        "yaf",  # y 軸刻度
        "zaf+lElevation (m)",  # 新增：垂直軸刻度與單位
        "+tTaiwan: 3D relief",
    ],
)


# 3. 顯示圖片
fig.colorbar(position="JBC+w10c/0.35c+h+o0c/1.5c", frame=["xaf", "y+lElevation (m)"], perspective=[180, 90])
fig.show()


### 用拉桿旋轉 3D 地形
拖動拉桿，觀察不同視角下的地形。完成環境設置後即可執行下格，程式會自行載入地形。

- **方位角**：繞地形旋轉，看看台灣的不同側面。
- **仰角**：由低角度斜看，逐漸拉高到接近俯視。

放開拉桿後才重新繪圖，避免拖動時連續計算。這是互動更新的靜態圖片，不是即時 3D 模型；需要運作中的 Colab／Jupyter，GitHub 靜態預覽不能操作。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [widgets.interact()](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html) | 將拉桿連到繪圖函式 | 函式參數與控制項的對應 |
| [IntSlider](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#IntSlider) | 建立整數拉桿 | `min`、`max`、`step`、`continuous_update` |
| [fig.grdview()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdview.html) | 依拉桿值重畫地形 | `perspective`、`zsize` |


In [ ]:
import pygmt
import ipywidgets as widgets


# 1. 本格自行載入地形；拉動滑桿時不重新下載
grid = pygmt.datasets.load_earth_relief(
    resolution="02m",
    region=[119, 123, 21, 26],
    registration="gridline",
)


# 2. 把原本的 3D 繪圖包成函式，讓拉桿傳入角度
def rotate_terrain(azimuth, elevation):
    fig = pygmt.Figure()

    fig.grdview(
        grid=grid,  # 沿用：已下載的地形，不重複下載
        region=[119, 123, 21, 26, -8000, 4000],
        projection="M12c",
        perspective=[azimuth, elevation],  # 改動：由拉桿決定視角
        zsize="2.4c",
        surftype="s",
        cmap="geo",
        frame=["xaf", "yaf", "zaf+lElevation (m)", "+tTaiwan relief"],
    )

    fig.colorbar(position="JBC+w8c/0.3c+h+o0c/1.5c", frame=["xaf", "y+lElevation (m)"], perspective=[180, 90])
    fig.show(dpi=100)  # 降低預覽解析度，加快更新


# 3. 每個拉桿對應函式的一個參數；放開滑鼠後才更新
widgets.interact(
    rotate_terrain,
    azimuth=widgets.IntSlider(
        value=135,
        min=0,
        max=360,
        step=5,
        description="方位角",
        continuous_update=False,
    ),
    elevation=widgets.IntSlider(
        value=35,
        min=10,
        max=85,
        step=5,
        description="仰角",
        continuous_update=False,
    ),
);


### 地形圖的圖說

GMT earth relief 2 角分網格，範圍 119–123°E、21–26°N。前三張依序呈現高程、加入方位角 −45° 的模擬照光、再疊加每 1000 m 的等高線（每 2000 m 標註）。色條高程以公尺表示，負值為海底。3D 圖使用相同區域，垂直尺寸為視覺設定，不能直接量取真實坡度；拉桿調整的是觀看方向。

修改解析度、照光、等高距或視角後，請同步修改圖說。


## 資料來源與版本

- [原始課程參考 Notebook](https://github.com/oceanicdayi/plot_plate_boundary_pygmt/blob/main/pygmt_plot_plate_boundary.ipynb)
- [GMT 全球地形資料](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html)：PyGMT 載入，首次使用需連網。
- [PyGMT 0.17 安裝文件](https://www.pygmt.org/v0.17.0/install.html)


| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 本課使用的真實事件資料 | 查詢條件包含在下載網址中 |

